# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
- Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- Citation: _Kulka, M., Rodriguez Miera, S., and Marcet-Palacios, M. 2026 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers._

<br/>

In [ ]:
# Ensure mlcroissant is installed in the current environment
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
CROISSANT_URL = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(CROISSANT_URL)

# Access metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")


## 2. Data Overview
Explore available record sets, fields, and their `@id`s.

We will examine the top-level record sets, their fields, and field IDs. All entities are referenced by their `@id`.

In [ ]:
# Get available record set @ids
record_sets = dataset.metadata.recordSet
if not record_sets or len(record_sets) == 0:
    # Defensive: some Croissant versions might list record sets under 'recordSet' key as a list or dict
    print("No record sets found via 'recordSet'. Attempting to list them via dataset.record_set_ids...")
    record_set_ids = dataset.record_set_ids()
else:
    record_set_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else str(rs) for rs in record_sets]

print(f"Record sets (@id): {record_set_ids}")

# For each record set, list fields and their @ids
for record_set_id in record_set_ids:
    print(f"\nRecord set: {record_set_id}")
    meta = dataset.metadata.find_by_id(record_set_id)
    if meta is None:
        continue
    # Get its fields
    fields = getattr(meta, 'field', [])
    # Defensive: 'field' may be list or dict. Normalize to list of dicts.
    if isinstance(fields, dict):
        fields = [fields]
    if isinstance(fields, list):
        for field in fields:
            fid = field.get('@id') if isinstance(field, dict) else str(field)
            fname = field.get('name') if isinstance(field, dict) and 'name' in field else ''
            print(f"  - Field @id: {fid}, name: {fname}")


## 3. Data Extraction
Extract data from a record set into a DataFrame for analysis. 

Use the record set and field `@id`s from overview above. If unsure, use `dataset.record_set_ids()` to enumerate.

We'll pick the first main record set which typically contains tabular data.

In [ ]:
# List available record sets explicitly
record_set_ids = dataset.record_set_ids()
print("RecordSet @ids detected:", record_set_ids)

# Select main record set for protein data (first one, but you may set by hand if there are several)
main_record_set_id = record_set_ids[0]

# Inspect a sample of records as dicts
print(f"\nSample records from {main_record_set_id}:")
for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
    print(json.dumps(rec, indent=2))
    if i > 1:
        break  # Show first 2 records only

# Load all records from the main record set to a DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df_main = pd.DataFrame(records)
print(f"\nColumns: {df_main.columns.tolist()}")
df_main.head()

## 4. Exploratory Data Analysis (EDA)

Let's perform basic EDA—filtering by a numeric field, normalizing, and grouping by a categorical field.

We'll select a numeric field (e.g., 'mw [kda]' or 'count') and a group field (e.g., 'description' or another category). Field names may require adjustment per available columns.

In [ ]:
# Inspect the available columns for numeric and group fields
print("Available columns:", df_main.columns.tolist())

# Choose a numeric field; some common ones in proteomics are 'mw [kda]', 'coverage [%]', or 'count'.
# Adjust field names as appropriate per actual DataFrame columns.
# Let's select 'mw [kda]' if present, otherwise fall back to a likely numeric field.
numeric_field_candidates = [col for col in df_main.columns if 'mw' in col.lower() or '%' in col or 'count' in col.lower() or 'abundance' in col.lower()]
if len(numeric_field_candidates) == 0:
    print("No numeric field candidates found. Please inspect df_main.columns.")
    numeric_field = None
else:
    numeric_field = numeric_field_candidates[0]  # pick first
print(f"Using numeric_field: {numeric_field}")

# Pick a group field: 'description' or any categorical field
group_field_candidates = [col for col in df_main.columns if 'desc' in col.lower() or 'group' in col.lower() or 'category' in col.lower()]
group_field = group_field_candidates[0] if len(group_field_candidates) else None
print(f"Using group_field: {group_field}")

if numeric_field:
    # Make sure the numeric field is numeric type (pandas may load as string)
    df_main[numeric_field] = pd.to_numeric(df_main[numeric_field], errors='coerce')
    # Filter
    threshold = df_main[numeric_field].mean() if df_main[numeric_field].notnull().any() else 0
    filtered_df = df_main[df_main[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows.")
    display(filtered_df.head(3))

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} (showing rows {filtered_df.index[:3].tolist()}):")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group and aggregate
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
        print(f"\nMean {numeric_field} grouped by {group_field}:")
        display(grouped_df.head())


## 5. Visualization

Visualize the distribution of the selected numeric field and its normalized counterpart. We'll also produce a boxplot grouped by the group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df_main[numeric_field].dropna(), kde=True, ax=ax[0], color='skyblue')
    ax[0].set_title(f"Distribution of {numeric_field}")
    if f"{numeric_field}_normalized" in filtered_df.columns:
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, ax=ax[1], color='orange')
        ax[1].set_title(f"Normalized {numeric_field} (filtered)")
    plt.tight_layout()
    plt.show()

    # Boxplot by group
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

This notebook demonstrated how to use `mlcroissant` to:
- Load and inspect a FAIR²-compliant Croissant dataset
- List record sets and fields by `@id`
- Extract records into a DataFrame for processing
- Analyze and visualize a quantitative field, including normalization and grouping

You can adapt the filter, grouping, and visualizations above to suit your research goals using precise record set, field, and column `@id`s as given by the Croissant schema.

<!-- Add further questions, notes, or suggestions for deeper analyses here. -->